# 17 — Gaussian Naive Bayes sul test ufficiale

Training sul **100% di KDDTrain+** (125973 righe), valutazione su **KDDTest+**
(22544 righe). Nessun train_test_split e nessun riuso dei modelli degli holdout.
var_smoothing=1e-9; deterministico su questo training.

Una configurazione congelata in T28, nessun tuning sul test. Le vecchie prove80/20
rimangono validation interna separata. Ogni esecuzione completa comporta un fit.
Il test è già stato consultato in T31: manteniamo il protocollo senza adattarlo agli esiti.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import sys
import warnings
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_recall_fscore_support,
)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src/data.py').is_file()
sys.path.insert(0, str(ROOT))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import AUTHOR_ATTACK_TO_FAMILY, FAMILY_ORDER

OUTPUT = ROOT / 'artifacts/gaussian_naive_bayes_official_test'
OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. Protocollo e dati

CSV senza intestazione nella cartella del progetto. Verifichiamo identità dei
file, versioni e schema prima dell'esperimento. Non creiamo un nuovo split.

In [ ]:
protocol_path = ROOT / 'configs/official_test_v1.json'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
protocol_hash = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
assert protocol['label_profile'] == 'author_75b8f8b'
assert protocol['features'] == list(FEATURE_COLUMNS)
assert protocol['family_order'] == list(FAMILY_ORDER)
assert protocol['split'] is None and protocol['test_tuning_allowed'] is False
versions = {name: importlib.metadata.version(name) for name in protocol['versions']}
assert versions == protocol['versions'], 'Versioni diverse dal protocollo'
for relative_path, expected in protocol['input_sha256'].items():
    assert hashlib.sha256((ROOT / relative_path).read_bytes()).hexdigest() == expected

string_types = {name: 'string' for name in (*CATEGORICAL_COLUMNS, 'attack_label')}
train = pd.read_csv(ROOT / 'data/raw/KDDTrain+.csv', header=None, names=COLUMN_NAMES, dtype=string_types)
test = pd.read_csv(ROOT / 'data/raw/KDDTest+.csv', header=None, names=COLUMN_NAMES, dtype=string_types)
numeric_columns = [name for name in FEATURE_COLUMNS if name not in CATEGORICAL_COLUMNS]
for name, frame in [('train', train), ('test', test)]:
    assert len(frame) == protocol['expected_rows'][name]
    assert frame.columns.tolist() == list(COLUMN_NAMES)
    assert not frame.isna().any().any()
    assert np.isfinite(frame[numeric_columns + ['difficulty']].to_numpy()).all()

## 2. Feature e target

Solo le 41 feature entrano nel modello. Target: cinque famiglie secondo il codice
autore, con worm → R2L e httptunnel → U2R. Le vecchie valutazioni restano invariate.
`difficulty` e il nome dell'attacco non sono predittori.

In [ ]:
for name, frame in [('train', train), ('test', test)]:
    unknown = ~frame['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY)
    if unknown.any():
        raise ValueError(f"Etichette non mappate in {name}: {frame.loc[unknown, 'attack_label'].tolist()}")
y_train = train['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).rename('family')
y_test = test['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).rename('family')
X_train = train.loc[:, list(FEATURE_COLUMNS)].copy()
X_test = test.loc[:, list(FEATURE_COLUMNS)].copy()
display(pd.DataFrame({'training': y_train.value_counts(), 'test': y_test.value_counts()}).reindex(FAMILY_ORDER))

## 3. Preprocessing — solo training

One-hot delle tre categoriche. StandardScaler per le 38 numeriche, appreso sul solo training.
`fit_transform` vede X_train; `transform` applica lo stesso preprocessing al test.
Target e difficulty sono esclusi. Le categorie ignote diventano blocchi di zeri.

In [ ]:
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), list(CATEGORICAL_COLUMNS)),
    ('numeric', StandardScaler(), numeric_columns),
])
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
encoder = preprocessor.named_transformers_['categorical']
for column, categories in zip(CATEGORICAL_COLUMNS, encoder.categories_):
    assert set(categories) == set(X_train[column])
assert X_train_encoded.shape[0] == len(X_train)
assert X_train_encoded.shape[1] == X_test_encoded.shape[1]
assert np.isfinite(X_train_encoded).all() and np.isfinite(X_test_encoded).all()
scaler = preprocessor.named_transformers_['numeric']
np.testing.assert_allclose(scaler.mean_, X_train[numeric_columns].mean().to_numpy())
np.testing.assert_allclose(scaler.var_, X_train[numeric_columns].var(ddof=0).to_numpy())
assert scaler.n_samples_seen_ == len(X_train)


## 4. Configurazione congelata

var_smoothing=1e-9; deterministico su questo training.
Confrontiamo tutti i parametri prima di eseguire il training.

In [ ]:
model = GaussianNB(priors=None, var_smoothing=1e-9)
actual_parameters = json.loads(json.dumps(model.get_params()))
assert actual_parameters == protocol['models']['gaussian_naive_bayes']['parameters'], 'configurazione diversa dal protocollo'

## 5. Training e diagnostica

Un fit sul training completo. Registriamo tempi, avvisi e convergenza senza cambiare
i parametri dopo il risultato. Gli avvisi sono parte dell'evidenza, non vengono nascosti.

In [ ]:
fit_start = perf_counter()
with warnings.catch_warnings(record=True) as captured_warnings:
    warnings.simplefilter('always')
    model.fit(X_train_encoded, y_train)
fit_seconds = perf_counter() - fit_start
fit_warnings = [{'category': w.category.__name__, 'message': str(w.message)} for w in captured_warnings]
diagnostics = {}
print(f'Training su {len(X_train):,} righe; tempo: {fit_seconds:.2f}s')
display(diagnostics)
display(fit_warnings)

## 6. Predizione sul test ufficiale

Applichiamo il modello già addestrato. Non scegliamo parametri in base all'esito.

In [ ]:
predict_start = perf_counter()
y_pred = model.predict(X_test_encoded)
predict_seconds = perf_counter() - predict_start
assert len(y_pred) == len(y_test)

## 7. Metriche globali e per famiglia

Macro F1 assegna lo stesso peso alle cinque famiglie; weighted F1 pesa il supporto
ed è la metrica aggregata usata nel confronto con il paper. `error_rate` è 1−recall:
la quota di esempi di una classe assegnati altrove. Precision e falsi positivi
servono per leggere anche gli allarmi sbagliati.

In [ ]:
cm_counts = confusion_matrix(y_test, y_pred, labels=FAMILY_ORDER)
cm_normalized = confusion_matrix(y_test, y_pred, labels=FAMILY_ORDER, normalize='true')
precision, recall, class_f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=FAMILY_ORDER, zero_division=0,
)
class_metrics = pd.DataFrame({
    'family': FAMILY_ORDER,
    'precision': precision,
    'recall': recall,
    'f1': class_f1,
    'support': support,
    'true_positives': np.diag(cm_counts),
    'false_positives': cm_counts.sum(axis=0) - np.diag(cm_counts),
    'false_negatives': cm_counts.sum(axis=1) - np.diag(cm_counts),
    'error_rate': 1 - recall,
})
metrics = {
    'macro_f1': float(f1_score(y_test, y_pred, labels=FAMILY_ORDER, average='macro', zero_division=0)),
    'weighted_f1': float(f1_score(y_test, y_pred, labels=FAMILY_ORDER, average='weighted', zero_division=0)),
    'accuracy': float(accuracy_score(y_test, y_pred)),
}

In [ ]:
display(pd.Series(metrics, name='test ufficiale').to_frame().round(4))
display(class_metrics.round(4))

## 8. Confusion matrix

Righe = famiglia vera, colonne = famiglia predetta. A destra ogni riga somma a 1:
la diagonale è il recall della famiglia. A sinistra vediamo i conteggi effettivi,
necessari per valutare il peso degli errori sulle classi rare.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm_counts, display_labels=FAMILY_ORDER).plot(ax=axes[0], cmap='Blues', values_format='d', colorbar=False)
ConfusionMatrixDisplay(cm_normalized, display_labels=FAMILY_ORDER).plot(ax=axes[1], cmap='Blues', values_format='.3f', colorbar=False)
axes[0].set_title('Test ufficiale — conteggi')
axes[1].set_title('Test ufficiale — normalizzata per classe vera')
for axis in axes:
    axis.set_xlabel('Famiglia predetta')
    axis.set_ylabel('Famiglia vera')
fig.tight_layout()
fig.savefig(OUTPUT / 'confusion_matrices.png', dpi=160, bbox_inches='tight')
plt.show()

## 9. Salvataggio delle evidenze

Predizioni, target e indici originali, configurazione, hash, versioni e diagnostica
sono salvati in una cartella dedicata. Ricontrollare le metriche non richiede training.

In [ ]:
np.savez_compressed(
    OUTPUT / 'predictions.npz',
    test_indices=test.index.to_numpy(),
    truth=y_test.to_numpy(dtype=str),
    predicted=np.asarray(y_pred, dtype=str),
)
class_metrics.to_csv(OUTPUT / 'class_metrics.csv', index=False)
pd.DataFrame(cm_counts, index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(OUTPUT / 'confusion_counts.csv', index_label='true_family')
pd.DataFrame(cm_normalized, index=FAMILY_ORDER, columns=FAMILY_ORDER).to_csv(OUTPUT / 'confusion_normalized.csv', index_label='true_family')
results = {
    'schema_version': 1,
    'model_id': 'gaussian_naive_bayes_official_test',
    'evaluation_set': 'KDDTest+ full, frozen configuration',
    'protocol_id': protocol['protocol_id'],
    'protocol_sha256': protocol_hash,
    'input_sha256': protocol['input_sha256'],
    'label_profile': protocol['label_profile'],
    'family_order': list(FAMILY_ORDER),
    'train_rows': len(y_train), 'test_rows': len(y_test),
    'raw_feature_count': len(FEATURE_COLUMNS),
    'encoded_feature_count': X_train_encoded.shape[1],
    'classifier_parameters': actual_parameters,
    'model_fits': 1,
    'seed': 1,
    'versions': versions,
    'preprocessing': 'one-hot and StandardScaler train-only',
    'encoder_categories': [values.tolist() for values in encoder.categories_],
    'numeric_columns': numeric_columns,
    'scaler_samples_seen': int(scaler.n_samples_seen_),
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_variance': scaler.var_.tolist(),
    'fit_seconds': fit_seconds, 'predict_seconds': predict_seconds,
    'diagnostics': diagnostics,
    'fit_warnings': fit_warnings,
    'metrics': metrics,
    'class_metrics': class_metrics.to_dict(orient='records'),
    'confusion_counts': cm_counts.tolist(),
    'confusion_normalized': cm_normalized.tolist(),
    'predictions_sha256': hashlib.sha256((OUTPUT / 'predictions.npz').read_bytes()).hexdigest(),
    'test_consulted': True,
}
(OUTPUT / 'results.json').write_text(json.dumps(results, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Evidenze di questo modello salvate.')

## 10. Interpretazione e limiti

Una sola configurazione e un solo seed dove applicabile: non stimiamo variabilità
o intervalli di confidenza. Il confronto con il paper è descrittivo, perché le
configurazioni e il preprocessing differiscono. Conserviamo i duplicati e le 654
feature-vector distinte condivise tra train e test, come dichiarato nel protocollo.

Leggere recall e precision insieme, soprattutto per U2R/R2L: alto recall con molti
falsi allarmi non implica un buon classificatore. I risultati saranno confrontati
con gli altri modelli sulle stesse righe e con lo stesso profilo di etichette.

## Risultato misurato e ricontrollato

Macro F1 **0.3352**, weighted F1 **0.4923**, accuracy **47.37%**.
Recall R2L **33.84%**, U2R **32.00%**;
precision R2L **15.89%**, U2R **6.15%**.

Un fit sul training completo. Metriche ricontrollate dalle predizioni senza refit.
Confronto con gli altri modelli e limiti nel notebook21 e nel report
`docs/OFFICIAL_BENCHMARK_REPORT.md`. Nessun parametro modificato dopo il test.
